# Demo 4: Monitoring Spectral Values During Training


Callbacks help monitor Lipschitz constraints. The `MonitorCallback` logs spectral statistics, while `CondenseCallback` can keep weights condensed throughout optimisation.


In [ ]:
import tempfile
import torch
from torch.utils.data import DataLoader, TensorDataset
from deel.lip.callbacks import CondenseCallback, MonitorCallback
from deel.lip.layers import SpectralDense, GroupSort2
from deel.lip.losses import TauCategoricalCrossentropy
from deel.lip.model import Sequential


In [ ]:
torch.manual_seed(3)
inputs = torch.randn(128, 16)
labels = torch.randint(0, 4, (128,))
dataset = TensorDataset(inputs, labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [ ]:
model = Sequential(
    SpectralDense(16, 32, activation="relu", use_bias=False),
    GroupSort2(),
    SpectralDense(32, 4, use_bias=False),
)
model.steps_per_epoch = len(loader)
loss_fn = TauCategoricalCrossentropy(tau=1.5)
optimizer = torch.optim.SGD(model.parameters(), lr=5e-3)
callbacks = [
    CondenseCallback(on_epoch=True, on_batch=False),
    MonitorCallback(monitored_layers=["0"], logdir=tempfile.mkdtemp(), target="kernel"),
]
for cb in callbacks:
    cb.set_model(model)


In [ ]:
def one_hot(targets, num_classes):
    return torch.nn.functional.one_hot(targets, num_classes=num_classes).float()

for epoch in range(3):
    total_loss = 0.0
    for batch_idx, (batch_inputs, batch_labels) in enumerate(loader):
        optimizer.zero_grad()
        logits = model(batch_inputs)
        loss = loss_fn(one_hot(batch_labels, 4), logits)
        loss.backward()
        optimizer.step()
        for cb in callbacks:
            cb.on_train_batch_end(batch_idx)
        total_loss += loss.item() * batch_inputs.size(0)
    for cb in callbacks:
        cb.on_epoch_end(epoch)
    print(f"Epoch {epoch + 1}: loss={(total_loss / len(dataset)):.4f}")


The temporary directory with TensorBoard logs can be inspected to verify that spectral values remain bounded during training.
